# TrialMatch — online GPU run (Kaggle or Colab)
Thin wrapper: all logic is in the repository's scripts.

* **Kaggle:** Settings → Accelerator → GPU T4 x2 (one GPU is used), Internet → On.
* **Colab:** Runtime → Change runtime type → T4 GPU.

T4 has no bf16, so training automatically uses fp16 autocast + GradScaler.

In [ ]:
# Option A: clone from GitHub (set the URL). Option B: upload trialmatch.zip and set ZIP_PATH instead.
REPO_URL = "https://github.com/Abubakar7-dotcom/Gen_AI_A2.git"
ZIP_PATH = None   # e.g. "/kaggle/input/trialmatch/trialmatch.zip" or "/content/trialmatch.zip"
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Pick a writable working folder: Kaggle -> /kaggle/working, Colab -> /content, else current folder.
WORK = next((d for d in ("/kaggle/working", "/content") if os.path.isdir(d)), os.getcwd())
print("working folder:", WORK)
os.chdir(WORK)
if ZIP_PATH:
    !rm -rf trialmatch && unzip -q $ZIP_PATH -d .
else:
    !rm -rf trialmatch && git clone -q $REPO_URL trialmatch
os.chdir(f"{WORK}/trialmatch")
!ls

In [ ]:
# Keep the platform's CUDA build of torch; install everything else at the pinned versions.
!grep -v '^torch==' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
# Colab ships torchao 0.10, which peft 0.21 rejects at LoRA creation; this project does not use torchao.
!pip uninstall -y -q torchao
!python -c "import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.get_device_name(0))"

## Run all experiments
Data split + tests → base-model validation → 3 LoRA trainings → fine-tuned validation.
Logs: `results/logs/*.log`, measured step timings: `results/logs/timings.json`.
The training log prints `step k/114 ... (elapsed s)`, so total training time = time per step × 114.

In [ ]:
!python scripts/run_gpu_experiments.py

## Download results and adapters (Kaggle: Output tab; Colab: Files panel)

In [ ]:
!cd .. && zip -qr trialmatch_run.zip trialmatch/results trialmatch/data/split_manifest.json trialmatch/outputs/*/best && ls -lh trialmatch_run.zip